# Projective geometry: meets, joins, and rigid-motion interpolation

Work with lines and points in **PGA2**, then planes, lines, and points in
**PGA3**. Both figures have interpolation sliders;
the 3D figure can also be rotated and zoomed.

Run all cells in order. The PGA types provide the exterior product `|`,
the regressive product `&`, and the Poincaré duals `ldual()`/`rdual()`
alongside the geometric product `*`. The small [`_geometry.py`](_geometry.py)
helpers construct points, lines, and planes and decode point coordinates;
`x.grade[k]` projects onto grade $k$.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.offline import init_notebook_mode
from algpy.algebras import pga2, pga3
from _geometry import pga_point, pga_hyperplane, pga_coordinates
from _plotting import slider_controls

init_notebook_mode(connected=False)
pio.templates.default = "plotly_white"

## Conventions: the null generator comes first

Write the first generator as $n$, with $n^2=0$; Euclidean generators square
to $+1$. We use the **dual PGA model**:

| Geometry | Grade 1 | Grade 2 | Grade 3 |
| --- | --- | --- | --- |
| PGA2 | line | point | pseudoscalar |
| PGA3 | plane | line | point |

A line/plane is $d\,n+\sum_i a_i e_i$, with equation $d+a\cdot x=0$.
A point is the left dual `(n + x e_1 + ...).ldual()` of its homogeneous
vector. Geometric meets (intersections) are exterior products, `a | b`.
Geometric joins (spans of points) are regressive products, `a & b`. The
Poincaré duals use signed basis complements rather than the metric, so they
work with the degenerate PGA metric, where the pseudoscalar has no inverse.

The operator names come from the lattice API: `|` is the lattice join, with
identity `bottom()`, and `&` the lattice meet, with identity `top()`. In
this dual model they intersect and span objects respectively.

In [ ]:
n2, x2, y2 = pga2.gens()
assert n2*n2 == pga2.zero()
I2 = n2*x2*y2
assert I2*I2 == pga2.zero()

A2 = pga_point(pga2, [-1.2, -.5])
B2 = pga_point(pga2, [.7, -.5])
C2 = pga_point(pga2, [-.5, .85])
edge2 = A2 & B2
cut2 = pga_hyperplane(pga2, [1, 0], .2)  # x = 0.2, i.e. x2 - .2*n2
meet2 = edge2 | cut2
np.testing.assert_allclose(pga_coordinates(meet2), [.2, -.5], atol=1e-13)
np.testing.assert_allclose(pga2.to_float(edge2 | A2), 0, atol=1e-13)
np.testing.assert_allclose(pga2.to_float(edge2 | B2), 0, atol=1e-13)
print("Line-line meet:", pga_coordinates(meet2))

## Motors from elementary functions

The exponential of a bivector generates a motion. A translation by $t$
and a counterclockwise planar rotation by $\theta$ are
\[
T=\exp(-nt/2),\qquad R=\exp(-\theta e_xe_y/2).
\]
Their product $M=TR$ acts on an object $X$ by $MXM^{-1}$. With this order,
rotation is followed by translation. We project back to the object's grade
to discard floating-point residue in other grades.

`one.mix(M, t)` interpolates in the algebra through powers and logarithms.
The examples choose rotations smaller than $\pi$, so the selected motor
representatives stay on a consistent principal branch.

In [ ]:
angle2 = .85
displacement2 = np.array([1., .65])
translation2 = np.exp(-.5*n2*(displacement2[0]*x2 + displacement2[1]*y2))
rotation2 = np.exp(-.5*angle2*x2*y2)
target2 = translation2*rotation2
rotation_matrix2 = np.array([[np.cos(angle2), -np.sin(angle2)],
                             [np.sin(angle2), np.cos(angle2)]])
for point in [A2, B2, C2]:
    moved = (target2*point*target2.inv()).grade[2]
    expected = rotation_matrix2 @ pga_coordinates(point) + displacement2
    np.testing.assert_allclose(pga_coordinates(moved), expected, atol=2e-12)

In [ ]:
def projective_scene2(t):
    motor = pga2.one().mix(target2, float(t))
    moved = [(motor*point*motor.inv()).grade[2] for point in [A2, B2, C2]]
    coordinates = np.array([pga_coordinates(point) for point in moved])
    joined_line = moved[0] & moved[1]
    intersection = pga_coordinates(joined_line | cut2)
    np.testing.assert_allclose(intersection[0], .2, atol=2e-11)
    for point in moved[:2]:
        np.testing.assert_allclose(pga2.to_float(joined_line | point), 0, atol=2e-11)
    polygon = np.vstack([coordinates, coordinates[0]])
    line_x = np.array([-1.6, 2.5])
    line_y = coordinates[0, 1] + (line_x-coordinates[0, 0])*(coordinates[1, 1]-coordinates[0, 1])/(coordinates[1, 0]-coordinates[0, 0])
    return [
        go.Scatter(x=line_x, y=line_y, mode="lines", opacity=.5,
                   line=dict(color="#2563eb", width=2), name="Joined infinite line AB"),
        go.Scatter(x=polygon[:, 0], y=polygon[:, 1], mode="lines+markers",
                   fill="toself", fillcolor="rgba(37,99,235,.10)",
                   line_color="#2563eb", name="Moved triangle"),
        go.Scatter(x=[intersection[0]], y=[intersection[1]], mode="markers",
                   marker=dict(size=12, color="#db2777"), name="Meet of AB and cut",
                   hovertemplate="x=%{x:.3f}<br>y=%{y:.3f}<extra>line-line meet</extra>"),
    ]

times = np.linspace(0, 1, 41)
fig = go.Figure(data=[go.Scatter(x=[.2, .2], y=[-1.4, 2.8], mode="lines",
                                 line=dict(color="#64748b", dash="dash"), name="Cut: x=0.2")]
                          + projective_scene2(0))
fig.frames = [go.Frame(name=str(index), data=projective_scene2(t), traces=[1, 2, 3])
              for index, t in enumerate(times)]
fig.update_layout(title="PGA2: join A and B, then meet the moving line with a fixed cut", height=570)
fig.update_xaxes(title="x", range=[-1.6, 2.5])
fig.update_yaxes(title="y", range=[-1.3, 2.7], scaleanchor="x", scaleratio=1)
slider_controls(fig, [f"{t:.2f}" for t in times]).show(auto_play=False)

## PGA3: three points join to a plane

Two points join to a line; that line joined with a third point gives a
plane. Two planes meet in a line, and three planes meet in a point.
Homogeneous scale and orientation do not affect the represented objects.

In [ ]:
n3, x3, y3, z3 = pga3.gens()
xy = np.array([[-1., -.7], [.9, -.5], [-.4, .9]])
xyz = np.column_stack([xy, .25*xy[:, 0] + .15*xy[:, 1] + .1])
points3 = [pga_point(pga3, row) for row in xyz]
line3 = points3[0] & points3[1]
plane3 = line3 & points3[2]
plane_x = pga_hyperplane(pga3, [1, 0, 0], .2)     # x = 0.2
plane_y = pga_hyperplane(pga3, [0, 1, 0], -.15)   # y = -0.15
vertical_line3 = plane_x | plane_y
intersection3 = plane3 | vertical_line3
np.testing.assert_allclose(pga_coordinates(intersection3), [.2, -.15, .1275], atol=1e-12)
for point in points3:
    np.testing.assert_allclose(pga3.to_float(plane3 | point), 0, atol=1e-12)
print("Plane-line meet:", pga_coordinates(intersection3))

In [ ]:
target3 = (np.exp(-.5*n3*(.55*x3 - .25*y3 + .7*z3))
           * np.exp(-.4*x3*y3))

def projective_scene3(t):
    motor = pga3.one().mix(target3, float(t))
    moved = [(motor*point*motor.inv()).grade[3] for point in points3]
    coordinates = np.array([pga_coordinates(point) for point in moved])
    plane = moved[0] & moved[1] & moved[2]
    intersection = pga_coordinates(plane | plane_x | plane_y)
    normal = np.cross(coordinates[1]-coordinates[0], coordinates[2]-coordinates[0])
    expected_z = (normal @ coordinates[0] - normal[0]*.2 + normal[1]*.15)/normal[2]
    np.testing.assert_allclose(intersection, [.2, -.15, expected_z], atol=2e-11)
    perimeter = np.vstack([coordinates, coordinates[0]])
    patch_xy = np.array([[-1.4, -1.4], [1.8, -1.4], [1.8, 1.8], [-1.4, 1.8]])
    patch_z = (normal @ coordinates[0] - patch_xy @ normal[:2])/normal[2]
    return [
        go.Mesh3d(x=patch_xy[:, 0], y=patch_xy[:, 1], z=patch_z,
                  i=[0, 0], j=[1, 2], k=[2, 3], opacity=.22, color="#2563eb", name="Joined plane patch"),
        go.Scatter3d(x=perimeter[:, 0], y=perimeter[:, 1], z=perimeter[:, 2],
                     mode="lines+markers", line=dict(color="#2563eb", width=5), name="A, B, C"),
        go.Scatter3d(x=[intersection[0]], y=[intersection[1]], z=[intersection[2]],
                     mode="markers", marker=dict(size=7, color="#db2777"), name="Plane-line meet"),
    ]

fig = go.Figure(data=[go.Scatter3d(x=[.2, .2], y=[-.15, -.15], z=[-.9, 1.9],
                                   mode="lines", line=dict(color="#64748b", width=5, dash="dash"),
                                   name="Meet of two fixed planes")]+projective_scene3(0))
fig.frames = [go.Frame(name=str(index), data=projective_scene3(t), traces=[1, 2, 3])
              for index, t in enumerate(times)]
fig.update_layout(title="PGA3: a moving joined plane meets a fixed line", height=640,
                  scene=dict(xaxis=dict(range=[-2.5, 2.5], title="x"),
                             yaxis=dict(range=[-2.5, 2.5], title="y"),
                             zaxis=dict(range=[-2.5, 2.5], title="z"), aspectmode="cube"))
slider_controls(fig, [f"{t:.2f}" for t in times]).show(auto_play=False)

**Try changing** the triangle vertices, target translation, or rotation.
Keep a finite intersection with the displayed cut. The translucent patch
represents the infinite joined plane; the three defining points need not
surround its intersection with the line.

Continue with `03_conformal_geometry.ipynb` for circles and spheres. CGA
uses a different representation and a nondegenerate metric dual.